## Attention backward test case 

In [81]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [82]:
# B = 2 , T = 3 , C = 4 , 3C = 12
x = torch.tensor([[[-2.0547,  0.2500,  0.2071,  0.7678,  0.9042, -1.4708,  0.2119,
          -0.3901, -0.6155, -0.0505, -0.9282,  1.4512],
         [-1.4378, -1.7302,  0.8048, -1.2562,  0.5207, -1.2107,  0.6358,
          -2.7949, -0.4698, -1.2076,  0.5352,  1.5699],
         [-1.2156, -1.1670, -0.6220, -1.6944, -0.6073, -0.0537, -0.1444,
          -0.6771,  0.2301,  0.6406, -1.0410,  1.8663]],

        [[ 0.3748, -1.4156, -0.0265, -1.1613, -0.1541,  0.8283, -1.1208,
          -0.2653,  1.7183,  0.4139, -1.0773, -1.6903],
         [-0.2480, -0.1178, -0.9540, -2.0342,  0.5077,  0.7330,  0.0504,
          -0.7498,  1.1569, -0.7619, -1.0578, -1.4162],
         [-0.7235,  0.0782, -0.6188, -1.4292, -0.4870,  0.3670,  1.3128,
          -0.5397,  0.9782, -1.0857, -1.7544,  1.5251]]])
x.requires_grad_(True)

tensor([[[-2.0547,  0.2500,  0.2071,  0.7678,  0.9042, -1.4708,  0.2119,
          -0.3901, -0.6155, -0.0505, -0.9282,  1.4512],
         [-1.4378, -1.7302,  0.8048, -1.2562,  0.5207, -1.2107,  0.6358,
          -2.7949, -0.4698, -1.2076,  0.5352,  1.5699],
         [-1.2156, -1.1670, -0.6220, -1.6944, -0.6073, -0.0537, -0.1444,
          -0.6771,  0.2301,  0.6406, -1.0410,  1.8663]],

        [[ 0.3748, -1.4156, -0.0265, -1.1613, -0.1541,  0.8283, -1.1208,
          -0.2653,  1.7183,  0.4139, -1.0773, -1.6903],
         [-0.2480, -0.1178, -0.9540, -2.0342,  0.5077,  0.7330,  0.0504,
          -0.7498,  1.1569, -0.7619, -1.0578, -1.4162],
         [-0.7235,  0.0782, -0.6188, -1.4292, -0.4870,  0.3670,  1.3128,
          -0.5397,  0.9782, -1.0857, -1.7544,  1.5251]]], requires_grad=True)

In [83]:
def causal_attention(query, key, value):
    # query, key, value have shape: (batch_size, num_heads, seq_len, head_dim)
    seq_len = query.size(-2)
    
    # SDPA computes: softmax((Q @ K.T) / sqrt(d_k)) @ V with a causal mask
    output = F.scaled_dot_product_attention(query, key, value, is_causal=True)
    return output

In [84]:
q11 = x[0,:,:2]
q12 = x[0,:,2:4]
k11 = x[0,:,4:6]
k12 = x[0,:,6:8]
v11 = x[0,:,8:10]
v12 = x[0,:,10:12]
out11 = causal_attention(q11,k11,v11)
out12 = causal_attention(q12,k12,v12)
out1 = torch.concat((out11,out12),dim=-1)

q21 =x[1,:,:2]
q22 =x[1,:,2:4]
k21 =x[1,:,4:6]
k22 =x[1,:,6:8]
v21 =x[1,:,8:10]
v22 =x[1,:,10:12]
out21 = causal_attention(q21,k21,v21)
out22 = causal_attention(q22,k22,v22)
out2 = torch.concat((out21,out22),dim=-1)

out = torch.stack((out1,out2))
print(out.shape)
out.retain_grad()

torch.Size([2, 3, 4])


In [85]:
loss = out.sum()

In [86]:
loss.backward()

In [87]:
print (x.grad)
print (out.grad)

tensor([[[ 0.0000,  0.0000,  0.0000,  0.0000, -0.2070, -0.2612, -0.0354,
           0.2033,  1.7888,  1.7888,  1.1422,  1.1422],
         [ 0.0685, -0.0464,  0.0369, -0.2091,  0.6117,  0.6497, -0.0090,
          -0.3243,  0.8620,  0.8620,  1.7633,  1.7633],
         [-0.3912,  0.3958,  0.0792, -0.2847, -0.4047, -0.3885,  0.0444,
           0.1210,  0.3492,  0.3492,  0.0945,  0.0945]],

        [[ 0.0000,  0.0000,  0.0000,  0.0000, -0.3141, -0.0103,  0.1560,
           0.3516,  1.8761,  1.8761,  1.9344,  1.9344],
         [-0.2026,  0.0292,  0.0607, -0.0251,  0.1270,  0.0305,  0.0032,
           0.0161,  0.7205,  0.7205,  0.8786,  0.8786],
         [ 0.0393,  0.1260,  0.5267, -0.0294,  0.1870, -0.0202, -0.1592,
          -0.3678,  0.4034,  0.4034,  0.1870,  0.1870]]])
tensor([[[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]],

        [[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]]])
